# Pipeline de Machine Learning usando o df tratado numérico

Este notebook usa diretamente `obesity_tratada_numerica.csv`. Como as variáveis categóricas já foram convertidas em números, a pipeline não usa `OneHotEncoder`; ela usa apenas imputação, padronização e o modelo.

## 1. Importar bibliotecas

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

## 2. Carregar o df tratado numérico

In [ ]:
df = pd.read_csv("obesity_tratada_numerica.csv")
df.head()

## 3. Conferir dimensões e se ainda existe texto

In [ ]:
print(df.shape)
print(df.select_dtypes(include="object").columns)

## 4. Separar variáveis explicativas e alvo

In [ ]:
target = "obesity_level_encoded"
X = df.drop(columns=[target])
y = df[target].astype(int)

print("X:", X.shape)
print("y:", y.shape)

## 5. Separar treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## 6. Criar pré-processamento numérico

In [ ]:
preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## 7. Definir modelos

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=80, random_state=42, n_jobs=1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=80, random_state=42, n_jobs=1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM RBF": SVC(kernel="rbf", random_state=42),
    "Naive Bayes": GaussianNB(),
}

## 8. Treinar e comparar modelos

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
results = []
pipelines = {}

for model_name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"},
        n_jobs=1
    )

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    results.append({
        "modelo": model_name,
        "cv_accuracy_media": scores["test_accuracy"].mean(),
        "cv_f1_macro_media": scores["test_f1_macro"].mean(),
        "teste_accuracy": accuracy_score(y_test, pred),
        "teste_f1_macro": f1_score(y_test, pred, average="macro"),
        "teste_precision_macro": precision_score(y_test, pred, average="macro"),
        "teste_recall_macro": recall_score(y_test, pred, average="macro"),
    })
    pipelines[model_name] = pipe

results_df = pd.DataFrame(results).sort_values(
    by=["cv_f1_macro_media", "teste_f1_macro"],
    ascending=False
).reset_index(drop=True)

results_df

## 9. Escolher e avaliar o melhor modelo

In [ ]:
best_model_name = results_df.loc[0, "modelo"]
best_pipeline = pipelines[best_model_name]
best_pred = best_pipeline.predict(X_test)

print("Melhor modelo:", best_model_name)
print("Accuracy:", accuracy_score(y_test, best_pred))
print("F1 macro:", f1_score(y_test, best_pred, average="macro"))
print("Precision macro:", precision_score(y_test, best_pred, average="macro"))
print("Recall macro:", recall_score(y_test, best_pred, average="macro"))
print(classification_report(y_test, best_pred))

## 10. Matriz de confusão

In [ ]:
pd.DataFrame(confusion_matrix(y_test, best_pred))

## 11. Importância das variáveis

In [ ]:
model = best_pipeline.named_steps["model"]

if hasattr(model, "feature_importances_"):
    feature_importance = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    display(feature_importance.head(20))
else:
    print("Este modelo não possui feature_importances_.")

## 12. Salvar artefatos

In [ ]:
results_df.to_csv("comparacao_modelos_df_numerico.csv", index=False)
joblib.dump(best_pipeline, "melhor_pipeline_obesity_numerico.joblib")